# entitymatch - Example Usage

This notebook demonstrates how to use `entitymatch` to match entity records across two datasets.

In [ ]:
import pandas as pd
from entitymatch import match_entities, EntityMatcher
from entitymatch.clean import clean_name, prepare_dataframe
from entitymatch.match import load_model, encode_names

## 1. Create Sample Data

Let's create two dataframes representing company records from different sources.

In [ ]:
# Source A
df_a = pd.DataFrame({
    "id": ["A1", "A2", "A3", "A4", "A5"],
    "company_name": [
        "McDonald's Corporation",
        "International Business Machines",
        "Walmart Inc",
        "Johnson & Johnson",
        "The Coca-Cola Company",
    ],
    "city": ["Chicago", "Armonk", "Bentonville", "New Brunswick", "Atlanta"],
    "state": ["IL", "NY", "AR", "NJ", "GA"],
})

# Source B (with name variations)
df_b = pd.DataFrame({
    "id": ["B1", "B2", "B3", "B4", "B5", "B6"],
    "company_name": [
        "McDonalds Corp",
        "IBM",
        "Wal-Mart Stores",
        "Johnson and Johnson Inc",
        "Coca Cola Co",
        "Target Corporation",
    ],
    "city": ["Chicago", "Armonk", "Bentonville", "New Brunswick", "Atlanta", "Minneapolis"],
    "state": ["IL", "NY", "AR", "NJ", "GA", "MN"],
})

print("Source A:")
display(df_a)
print("\nSource B:")
display(df_b)

## 2. Name Cleaning

See how names are normalized before matching.

In [ ]:
for name in df_a["company_name"]:
    print(f"{name:40s} -> {clean_name(name)}")

## 3. Run Full Pipeline

Match the two datasets using `match_entities`.

In [ ]:
results = match_entities(
    df_left=df_a,
    df_right=df_b,
    left_name_col="company_name",
    right_name_col="company_name",
    left_id_col="id",
    right_id_col="id",
    left_city_col="city",
    right_city_col="city",
    left_state_col="state",
    right_state_col="state",
    threshold=0.50,
    auto_accept_threshold=0.80,
    show_progress=True,
)

results[["left_id", "right_id", "left_name", "right_name", "score", "accept_reason"]]

## 4. With LLM Validation (Optional)

To validate gray-zone matches with an LLM, set your API key and enable LLM validation.

In [ ]:
# Uncomment and set your API key:
# import os
# os.environ["OPENAI_API_KEY"] = "your-key-here"
#
# results_llm = match_entities(
#     df_left=df_a,
#     df_right=df_b,
#     left_name_col="company_name",
#     right_name_col="company_name",
#     left_id_col="id",
#     right_id_col="id",
#     left_city_col="city",
#     right_city_col="city",
#     left_state_col="state",
#     right_state_col="state",
#     use_llm=True,
#     llm_provider="openai",
#     llm_model="gpt-4o-mini",
# )
# results_llm[["left_id", "right_id", "score", "llm_match", "accept_reason"]]

## 5. Using Individual Components

You can also use each stage independently.

In [ ]:
# Prepare data
left = prepare_dataframe(df_a, name_col="company_name", city_col="city", state_col="state", id_col="id")
right = prepare_dataframe(df_b, name_col="company_name", city_col="city", state_col="state", id_col="id")

# Encode
model = load_model()
emb_left = encode_names(left["name_clean"], model=model, show_progress=False)
emb_right = encode_names(right["name_clean"], model=model, show_progress=False)

print(f"Encoded {len(emb_left)} left names and {len(emb_right)} right names")
print(f"Embedding dimension: {list(emb_left.values())[0].shape[0]}")